In [ ]:
!pip install -q torch torchvision facenet-pytorch albumentations opencv-python-headless scikit-image tqdm matplotlib

In [ ]:
import os
import cv2
import random
import numpy as np
import torch
from torch import nn, optim
from torch.utils.data import Dataset, DataLoader
from facenet_pytorch import MTCNN, InceptionResnetV1
import albumentations as A
from albumentations.pytorch import ToTensorV2
from tqdm import tqdm
from shutil import copy2
from skimage.metrics import peak_signal_noise_ratio as psnr
from skimage.metrics import structural_similarity as ssim
import matplotlib.pyplot as plt

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

## 1. Configuration

In [ ]:

SRC_FOLDER = "./images"                    # raw CelebA images
GENERALIZATION_FOLDER = "./other_faces"  
ALIGNED_FOLDER = "./aligned"
GEN_ALIGNED_FOLDER = "./aligned_generalization"
DATA_ROOT = "./data"

IMAGE_SIZE = 160
BATCH_SIZE = 8
PHASE1_EPOCHS = 5     
PHASE2_EPOCHS = 30    # restoration + identity loss
IDENTITY_LOSS_WEIGHT = 0.1

os.makedirs(ALIGNED_FOLDER, exist_ok=True)
os.makedirs(GEN_ALIGNED_FOLDER, exist_ok=True)

## 2. Face Detection & Alignment
MTCNN detects the face and aligns it using its 5 facial landmarks (eyes, nose, mouth corners),
producing a consistently cropped, centered face — this stabilizes training as required by the assignment.

In [ ]:
mtcnn = MTCNN(image_size=IMAGE_SIZE, margin=20, device=device)

def align_faces(src_folder, dst_folder):
    n_ok, n_fail = 0, 0
    for name in tqdm(os.listdir(src_folder), desc=f"Aligning {src_folder}"):
        try:
            path = os.path.join(src_folder, name)
            img = cv2.imread(path)
            if img is None:
                n_fail += 1
                continue
            img_rgb = img[:, :, ::-1]
            face, prob = mtcnn(img_rgb, return_prob=True)
            if face is None:
                n_fail += 1
                continue
            face = (face.permute(1, 2, 0).clamp(0, 1).cpu().numpy() * 255).astype(np.uint8)
            cv2.imwrite(os.path.join(dst_folder, name), face[:, :, ::-1])
            n_ok += 1
        except Exception as e:
            n_fail += 1
            print("Error:", name, e)
    print(f"Done. Aligned: {n_ok}, Skipped/failed: {n_fail}")

align_faces(SRC_FOLDER, ALIGNED_FOLDER)

## 3. Train / Validation / Test Split

In [ ]:
def make_splits(aligned_folder, data_root, train_ratio=0.75, val_ratio=0.15):
    all_images = [f for f in os.listdir(aligned_folder)
                  if f.lower().endswith((".jpg", ".jpeg", ".png"))]
    random.shuffle(all_images)

    n = len(all_images)
    train_end = int(n * train_ratio)
    val_end = int(n * (train_ratio + val_ratio))

    splits = {
        "train": all_images[:train_end],
        "val": all_images[train_end:val_end],
        "test": all_images[val_end:],
    }

    for split, files in splits.items():
        folder = os.path.join(data_root, split)
        os.makedirs(folder, exist_ok=True)
        for fname in files:
            copy2(os.path.join(aligned_folder, fname), folder)
        print(f"{split}: {len(files)} images")

    return splits

make_splits(ALIGNED_FOLDER, DATA_ROOT)

## 4. Augmentation & Degradation

Augmentation (applied to the *clean* target — flips, brightness/contrast, small rotation/shift)
is kept separate from **degradation** (applied to the model's *input* — this simulates the
corrupted images the model must learn to restore).

Each training sample gets **one** randomly chosen degradation type, matching the assignment
(downsampling **or** Gaussian noise), with motion blur added as a third option for the extra-credit requirement.

In [ ]:
augment = A.Compose([
    A.HorizontalFlip(p=0.5),
    A.RandomBrightnessContrast(p=0.3),
    A.ShiftScaleRotate(shift_limit=0.05, scale_limit=0.1, rotate_limit=10, p=0.5),
    ToTensorV2()
])

def motion_blur(img_uint8, kernel_size=9):
    k = kernel_size
    kernel = np.zeros((k, k))
    angle = random.randint(0, 180)
    kernel[k // 2, :] = np.ones(k)
    M = cv2.getRotationMatrix2D((k / 2, k / 2), angle, 1)
    kernel = cv2.warpAffine(kernel, M, (k, k))
    kernel /= kernel.sum()
    return cv2.filter2D(img_uint8, -1, kernel)

def degrade_downsample(img_float01):
    h, w = img_float01.shape[:2]
    scale = 1 / random.choice([1.2, 1.4])
    small = cv2.resize(img_float01, (max(1, int(w * scale)), max(1, int(h * scale))))
    return cv2.resize(small, (w, h))

def degrade_gaussian_noise(img_float01):
    noise_std = random.uniform(0.2, 0.5)
    noise = np.random.normal(0, noise_std, img_float01.shape)
    return np.clip(img_float01 + noise, 0, 1)

def degrade_motion_blur(img_float01):
    blurred = motion_blur((img_float01 * 255).astype(np.uint8))
    return blurred.astype(np.float32) / 255.0

def degrade(img_uint8):
    """Takes a uint8 RGB image, returns a degraded image in [0,1] float."""
    img = img_uint8.astype(np.float32) / 255.0
    choice = random.choice(["downsample", "noise", "blur"])
    if choice == "downsample":
        img = degrade_downsample(img)
    elif choice == "noise":
        img = degrade_gaussian_noise(img)
    else:
        img = degrade_motion_blur(img)
    return np.clip(img, 0, 1)

## 5. Dataset

**Fix vs. the draft notebooks:** both the clean and degraded tensors are explicitly normalized
to `[0, 1]` here (`/255.0`) — without this, the loss and PSNR/SSIM scores are computed against
a `[0,255]` target while the model outputs `sigmoid()` values in `[0,1]`, which silently breaks
training. The dataset also skips unreadable files instead of crashing the whole run.

In [ ]:
class FaceDataset(Dataset):
    def __init__(self, folder):
        self.paths = [os.path.join(folder, x) for x in os.listdir(folder)
                      if x.lower().endswith((".jpg", ".jpeg", ".png"))]
        self.paths.sort()
        if len(self.paths) == 0:
            raise RuntimeError(f"No images found in {folder}")

    def __len__(self):
        return len(self.paths)

    def _load(self, idx):
        img_bgr = cv2.imread(self.paths[idx])
        if img_bgr is None:
            return None
        return img_bgr[:, :, ::-1]  # BGR -> RGB

    def __getitem__(self, idx):
        img = self._load(idx)
        tries = 0
        while img is None and tries < 5:
            idx = (idx + 1) % len(self.paths)
            img = self._load(idx)
            tries += 1
        if img is None:
            raise RuntimeError("Too many unreadable images in a row.")

        degraded01 = degrade(img)  # float [0,1]

        clean_aug = augment(image=img)["image"].float() / 255.0
        bad_aug = augment(image=(degraded01 * 255).astype(np.uint8))["image"].float() / 255.0

        return bad_aug, clean_aug


train_loader = DataLoader(FaceDataset(os.path.join(DATA_ROOT, "train")),
                           batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
val_loader = DataLoader(FaceDataset(os.path.join(DATA_ROOT, "val")),
                         batch_size=BATCH_SIZE, num_workers=2)
test_loader = DataLoader(FaceDataset(os.path.join(DATA_ROOT, "test")),
                          batch_size=BATCH_SIZE, num_workers=2)

## 6. Model — U-Net Encoder–Decoder

In [ ]:
class ConvBlock(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, 3, 1, 1),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_ch, out_ch, 3, 1, 1),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
        )

    def forward(self, x):
        return self.net(x)


class UNet(nn.Module):
    """3-level U-Net. Deeper than a single-level toy U-Net, still light enough to train
    on a single GPU/Colab in reasonable time."""

    def __init__(self, base=64):
        super().__init__()
        self.e1 = ConvBlock(3, base)
        self.e2 = ConvBlock(base, base * 2)
        self.e3 = ConvBlock(base * 2, base * 4)
        self.pool = nn.MaxPool2d(2)

        self.bottleneck = ConvBlock(base * 4, base * 8)

        self.up3 = nn.ConvTranspose2d(base * 8, base * 4, 2, 2)
        self.d3 = ConvBlock(base * 8, base * 4)
        self.up2 = nn.ConvTranspose2d(base * 4, base * 2, 2, 2)
        self.d2 = ConvBlock(base * 4, base * 2)
        self.up1 = nn.ConvTranspose2d(base * 2, base, 2, 2)
        self.d1 = ConvBlock(base * 2, base)

        self.out = nn.Conv2d(base, 3, 1)

    def forward(self, x):
        e1 = self.e1(x)
        e2 = self.e2(self.pool(e1))
        e3 = self.e3(self.pool(e2))

        b = self.bottleneck(self.pool(e3))

        d3 = self.up3(b)
        d3 = self.d3(torch.cat([d3, e3], dim=1))
        d2 = self.up2(d3)
        d2 = self.d2(torch.cat([d2, e2], dim=1))
        d1 = self.up1(d2)
        d1 = self.d1(torch.cat([d1, e1], dim=1))

        return torch.sigmoid(self.out(d1))


model = UNet().to(device)
print(sum(p.numel() for p in model.parameters() if p.requires_grad), "trainable parameters")

## 7. Training — Phase 1: Restoration Loss Only
A short warm-up with plain L1 loss stabilizes the weights before adding the identity loss.

In [ ]:
l1_loss = nn.L1Loss()
optimizer = optim.Adam(model.parameters(), lr=1e-4)

def run_epoch(loader, train=True):
    model.train() if train else model.eval()
    total = 0.0
    ctx = torch.enable_grad() if train else torch.no_grad()
    with ctx:
        for bad, clean in tqdm(loader, leave=False):
            bad, clean = bad.to(device), clean.to(device)
            pred = model(bad)
            loss = l1_loss(pred, clean)
            if train:
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()
            total += loss.item()
    return total / len(loader)

for epoch in range(PHASE1_EPOCHS):
    train_loss = run_epoch(train_loader, train=True)
    val_loss = run_epoch(val_loader, train=False)
    print(f"[Phase 1] Epoch {epoch+1}/{PHASE1_EPOCHS} | Train L1: {train_loss:.4f} | Val L1: {val_loss:.4f}")

## 8. Training — Phase 2: Restoration + Identity Loss
Adds an identity/recognition loss based on the embedding distance from a pretrained face
recognition model (`InceptionResnetV1`, trained on VGGFace2 — a substitute for AdaFace, since
`facenet-pytorch` ships pretrained weights that work out of the box). This encourages the model
to preserve *who* the person is, not just pixel-level similarity.

In [ ]:
face_embedder = InceptionResnetV1(pretrained="vggface2").eval().to(device)
for p in face_embedder.parameters():
    p.requires_grad = False

def recognition_loss(pred, target):
    e1 = face_embedder(pred)
    e2 = face_embedder(target)
    return torch.mean((e1 - e2) ** 2)

optimizer = optim.Adam(model.parameters(), lr=1e-4)

def run_epoch_combined(loader, train=True):
    model.train() if train else model.eval()
    total_r, total_id, total = 0.0, 0.0, 0.0
    ctx = torch.enable_grad() if train else torch.no_grad()
    with ctx:
        for bad, clean in tqdm(loader, leave=False):
            bad, clean = bad.to(device), clean.to(device)
            pred = model(bad)

            loss_r = l1_loss(pred, clean)
            loss_id = recognition_loss(pred, clean)
            loss = loss_r + IDENTITY_LOSS_WEIGHT * loss_id

            if train:
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()

            total_r += loss_r.item()
            total_id += loss_id.item()
            total += loss.item()
    n = len(loader)
    return total / n, total_r / n, total_id / n

history = []
for epoch in range(PHASE2_EPOCHS):
    train_total, train_r, train_id = run_epoch_combined(train_loader, train=True)
    val_total, val_r, val_id = run_epoch_combined(val_loader, train=False)
    history.append((train_total, val_total))
    print(f"[Phase 2] Epoch {epoch+1}/{PHASE2_EPOCHS} | "
          f"Train: {train_total:.4f} (L1 {train_r:.4f} + Id {train_id:.4f}) | "
          f"Val: {val_total:.4f} (L1 {val_r:.4f} + Id {val_id:.4f})")

In [ ]:
plt.figure(figsize=(6, 4))
plt.plot([h[0] for h in history], label="train")
plt.plot([h[1] for h in history], label="val")
plt.xlabel("Epoch"); plt.ylabel("Combined loss"); plt.legend(); plt.title("Phase 2 training curve")
plt.show()

## 9. Evaluation — PSNR, SSIM, Embedding Distance (Test Set)

In [ ]:
@torch.no_grad()
def evaluate(loader, tag=""):
    model.eval()
    psnr_list, ssim_list, embed_list = [], [], []
    for bad, clean in tqdm(loader, desc=f"Evaluating {tag}", leave=False):
        bad, clean = bad.to(device), clean.to(device)
        pred = model(bad)

        for p, c in zip(pred.cpu().numpy(), clean.cpu().numpy()):
            p_img = p.transpose(1, 2, 0)
            c_img = c.transpose(1, 2, 0)
            psnr_list.append(psnr(c_img, p_img, data_range=1))
            ssim_list.append(ssim(c_img, p_img, channel_axis=2, data_range=1))

        embed_list.append(recognition_loss(pred, clean).item())

    print(f"--- {tag} ---")
    print(f"PSNR:              {np.mean(psnr_list):.3f} dB")
    print(f"SSIM:              {np.mean(ssim_list):.4f}")
    print(f"Embedding distance:{np.mean(embed_list):.4f}")
    return np.mean(psnr_list), np.mean(ssim_list), np.mean(embed_list)

test_metrics = evaluate(test_loader, tag="CelebA test set")

## 10. Generalization Test — a Different Face Dataset

The assignment requires checking whether the model generalizes beyond CelebA. Point
`GENERALIZATION_FOLDER` (set in section 1) at a **different** face dataset
(e.g. LFW, a personal photo folder, or any other face collection not used in training).

In [ ]:
if os.path.isdir(GENERALIZATION_FOLDER) and len(os.listdir(GENERALIZATION_FOLDER)) > 0:
    align_faces(GENERALIZATION_FOLDER, GEN_ALIGNED_FOLDER)
    gen_loader = DataLoader(FaceDataset(GEN_ALIGNED_FOLDER), batch_size=BATCH_SIZE)
    gen_metrics = evaluate(gen_loader, tag="Generalization set")

    print("\nComparison — CelebA test vs. generalization set:")
    print(f"{'Metric':<20}{'CelebA test':<15}{'Other dataset':<15}")
    print(f"{'PSNR (dB)':<20}{test_metrics[0]:<15.3f}{gen_metrics[0]:<15.3f}")
    print(f"{'SSIM':<20}{test_metrics[1]:<15.4f}{gen_metrics[1]:<15.4f}")
    print(f"{'Embedding dist.':<20}{test_metrics[2]:<15.4f}{gen_metrics[2]:<15.4f}")
else:
    print(f"Set GENERALIZATION_FOLDER to a real folder of face images to run this section "
          f"(currently: '{GENERALIZATION_FOLDER}').")

## 11. Qualitative Results — Degraded vs. Restored vs. Original

In [ ]:
@torch.no_grad()
def show_examples(loader, n=4):
    model.eval()
    bad, clean = next(iter(loader))
    bad, clean = bad.to(device), clean.to(device)
    pred = model(bad)

    fig, axes = plt.subplots(n, 3, figsize=(9, 3 * n))
    for i in range(n):
        for ax, img, title in zip(
            axes[i],
            [bad[i], pred[i], clean[i]],
            ["Degraded input", "Restored", "Original"],
        ):
            ax.imshow(img.cpu().permute(1, 2, 0).clamp(0, 1).numpy())
            ax.set_title(title)
            ax.axis("off")
    plt.tight_layout()
    plt.show()

show_examples(test_loader, n=4)

## 12. Save the Trained Model

In [ ]:
torch.save(model.state_dict(), "face_restoration_unet.pth")
print("Model saved to face_restoration_unet.pth")